In [1]:
import json
import os

from sklearn.model_selection import train_test_split

from datasets import load_dataset

dataset = load_dataset("QCRI/DisasterVQA", cache_dir= "./data/hf_cache")

dataset = dataset["train"]


c:\Users\Shourya\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
binary_samples = []

for sample in dataset:

    if sample["question_type"] != "Binary":
        continue

    answer = sample["groundtruth_answer"][0]

    if answer not in ["Yes","No"]:
        continue

    binary_samples.append(sample)

print(len(binary_samples))

2153


In [3]:
answers = [sample["groundtruth_answer"][0] for sample in binary_samples]

train_samples, val_samples = train_test_split(
    binary_samples,
    test_size=0.2,
    random_state=42,
    stratify=answers
)

print(f"Train samples: {len(train_samples)}")
print(f"Val samples: {len(val_samples)}")

Train samples: 1722
Val samples: 431


Making the word dictionary now

In [4]:
from collections import Counter

counter = Counter()

for sample in train_samples:
    question = sample["question"].lower()
    words = question.split()
    counter.update(words)

print("Unique words:", len(counter))

Unique words: 732


In [5]:
word2idx = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word in counter:
    word2idx[word] = len(word2idx)

idx2word = {v: k for k, v in word2idx.items()}

print("Vocab size:", len(word2idx))

Vocab size: 734


Creating encoders for all the questions

In [6]:
question_lengths = []

for sample in dataset:
    words = sample["question"].lower().split()
    question_lengths.append(len(words))

print("Max length:", max(question_lengths))
print("Avg length:", sum(question_lengths)/len(question_lengths))

Max length: 23
Avg length: 12.931895573212259


In [7]:
MAX_QUESTION_LEN = 23

def encode_question(question):

    words = question.lower().split()
    tokens = [word2idx.get(word, word2idx["<UNK>"]) for word in words]

    if len(tokens) < MAX_QUESTION_LEN:
        tokens += [word2idx["<PAD>"]] * (MAX_QUESTION_LEN - len(tokens))
    else:
        tokens = tokens[:MAX_QUESTION_LEN]

    return tokens

In [8]:
train_processed = []

for sample in train_samples:

    train_processed.append(
        {
            "image": sample["image"],
            "question": sample["question"],
            "question_tokens": encode_question(sample["question"]),
            "answer": 1 if sample["groundtruth_answer"][0] == "Yes" else 0
        }
    )

In [9]:
val_processed = []

for sample in val_samples:

    val_processed.append(
        {
            "image": sample["image"],
            "question": sample["question"],
            "question_tokens": encode_question(sample["question"]),
            "answer": 1 if sample["groundtruth_answer"][0] == "Yes" else 0
        }
    )

In [10]:
import pickle
import os

# os.makedirs("data", exist_ok=True)

# with open("data/train_processed.pkl", "wb") as f:
#     pickle.dump(train_processed, f)

# with open("data/val_processed.pkl", "wb") as f:
#     pickle.dump(val_processed, f)

# with open("data/word2idx.pkl", "wb") as f:
#     pickle.dump(word2idx, f)

# with open("data/idx2word.pkl", "wb") as f:
#     pickle.dump(idx2word, f)

# print("Saved successfully.")

In [11]:
import pickle
import os

with open("data/train_processed.pkl", "rb") as f:
    train_loaded = pickle.load(f)

print(len(train_loaded))
print(train_loaded[0])


1722
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1200x799 at 0x21806AC3BC0>, 'question': 'Are there any emergency responders present? Answer with Yes or No.', 'question_tokens': [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'answer': 1}
